# From a video frame to a count

**Author:** Antoine ([@antowan](https://github.com/antowan))

A camera looks at a street. You want to know **how many cars, people and trucks are in view**, not just draw boxes. This example follows one frame through `AIPV1EMI0001`, the AI Processor (AIP) you’ll build in the next notebook. There’s nothing to install or run here.

## Follow one frame

![One frame flows through five stages: camera or file, decode and batch frames with DeepStream, detect objects with nvinferserver and the parser, track objects with nvtracker, then count and publish with callback.py](../doc/assets/aip-pipeline.svg)

DeepStream decodes the video and groups frames from different inputs into a **batch**. Its `nvinferserver` plugin runs the detector through Triton. For each box, a small C++ **parser** reads the class and score and keeps only cars, people and trucks.

Next, `nvtracker` uses the NvDCF tracker to associate the same object across frames. If you draw a region or line in the station, the analytics stage can also count objects inside that region or crossing that line.

Finally, [`callback.py`](../callback.py) turns the metadata into counts by source. If a car stays in view for ten frames, that’s one car visible in each frame, **not ten different cars**. The dashboard can show the largest number visible at once in a time window; a line-crossing total answers a different question. See the [callback output with examples](../doc/USER_MANUAL.md#callback-output-metadata-contract).

## What you put in the package

The station builds the pipeline. Your job is to supply what it needs to run this detector:

| Part | Job |
|---|---|
| [`manifest.json`](../manifest.json) | Names the processor, its three classes, the detector configuration and the tracker. |
| Model, labels and parser | A TensorRT **engine** runs the network on the station GPU; the parser translates its output into detections that DeepStream understands. |
| Triton and DeepStream settings | Describe the tensors, resize frames to the model’s 640 × 640 input and select the parser. |
| Callback and dashboard | The callback reports the counts; the bundled Node-RED project can display them alongside the video. Dashboard setup is separate. |

You start with pretrained weights, export them to **ONNX** (a model graph), then compile that graph into an engine for the target GPU and TensorRT version. EAIS Tools packages the runtime files into an installable `.tgz`; training and ONNX export are not part of the running pipeline.

## What changes when you switch YOLO models?

One package contains **one** detector. Each [flavor](../flavors/) specifies the weights, export settings and model-specific configuration. The tracker, callback and dashboard stay the same because the parser presents the same three classes in the same order.

- **YOLO26n and YOLOv10n** return one tensor with up to 300 candidate rows per image. Each row contains a box, a score and a class. These end-to-end heads need no non-maximum suppression (NMS) step in the engine.
- **YOLOv7-tiny** can propose several boxes for one object. Its engine includes an NMS plugin that suppresses overlapping proposals and returns four tensors: a count, boxes, scores and classes.

That difference is why swapping a weights file alone would not work: the exported outputs must agree with the Triton configuration and parser.

## Your first build

On a development station, open **one** build notebook: [YOLO26n (recommended)](03-build-yolo26.ipynb), [YOLOv10n](02-build-yolov10.ipynb) or [YOLOv7-tiny](01-build-yolov7.ipynb). The first milestone is an archive; the second is a test video producing callback records with detections.

A detection on a sample video is not an accuracy score. To choose a model, read the [accuracy and latency comparison](04-compare-models.ipynb). To size a pipeline, use the [measured RTSP camera counts](../doc/benchmarks/orin-nx-16gb-jp6.0/capacity.md), not the maximum batch size in the manifest.